# 02 — Klasik Baseline (YOL C): local-max detection + Hungarian linking

**Sıfır pip.** Ultrack/MONAI kurulamadığı ve GPU olmadığı için tek uygulanabilir yol.

> ### ⚠️ SUBMIT KURALLARI — okumadan çalıştırma
> - **Internet KAPALI olmalı** (Settings → Internet = Off). `pip install` YOK.
> - **`erdeemt/cell-tracking-libs` dataset'i EKLİ olmalı** — `zarr` bu ortamda yok,
>   veri onsuz okunamaz. `sys.path.append` ile yükleniyor (ilk hücre).
> - Bu bir **code competition**: notebook, **gizli test setiyle** yeniden çalıştırılır.
> - Bu yüzden **test isimleri hardcode EDİLMEZ** — `test/*.zarr` runtime'da glob'lanır.
> - Çıktı dosyası **`submission.csv`** olmalı (`/kaggle/working/`).

**Pipeline:** normalize (attrs quantile) → anizotropik local-max detection → Hungarian linking (8 µm) → CSV

**EDA'dan gelen parametreler** (bkz. `docs/EDA_BULGULARI.md`):
- ölçek `(Z,Y,X) = (1.625, 0.40625, 0.40625)` µm — dosyadan okunur
- hareket: medyan ~2–3 µm, 99p ~8 µm → **linking gate 8 µm**
- çekirdek ~10 µm çap → bastırma yarıçapı ~5 µm
- **`T_pred ≈ 200/kare` hedefle** (T_true ~213; fazla-tahmin cezası)
- bölünme %0.11 ve skorun %10'u → **baseline'da atlanıyor**

In [ ]:
# === ZORUNLU: zarr bu ortamda YOK, utility dataset'ten geliyor ===
# Dataset: erdeemt/cell-tracking-libs  ->  notebook'a EKLI olmali (private re-run'da da gerekli)
# append! insert(0) DEGIL: pylibs'te numpy 2.5.1 var, ortamin 2.0.2'sini golgelememeli
# (scipy/skimage/torch 2.0.2'ye karsi derlenmis).
import sys
from pathlib import Path

LIBS = Path('/kaggle/input/datasets/erdeemt/cell-tracking-libs/pylibs')
if not LIBS.exists():                       # mount yolu degisirse bul
    hits = list(Path('/kaggle/input').rglob('pylibs'))
    assert hits, 'cell-tracking-libs dataset i notebook a ekli degil!'
    LIBS = hits[0]
sys.path.append(str(LIBS))

import json, time
import numpy as np
import pandas as pd
import zarr
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

print('zarr ', zarr.__version__, 'from', 'pylibs' if 'pylibs' in zarr.__file__ else zarr.__file__)
print('numpy', np.__version__, '(2.0.2 = ortamdan, dogru)')
assert np.__version__.startswith('2.0'), f'numpy golgelendi: {np.__version__} — insert(0) mi kullandin?'

ROOT = Path('/kaggle/input/competitions/biohub-cell-tracking-during-development')
OUT  = Path('/kaggle/working')

# --- Ayarlanabilir parametreler (EDA + 1. kosunun olcumleri) ---
LINK_GATE_UM   = 8.0    # kare-arasi maksimum yer degistirme (hareket 99p ~8 um)
SUPPRESS_UM    = 5.0    # local-max bastirma yaricapi (cekirdek ~10 um cap)
SMOOTH_UM      = 1.5    # gaussian sigma (um)
NOISE_PCT      = 50     # KARE-BASINA gurultu tabani persentili (sabit esik DEGIL!)
BORDER_MARGIN  = 1      # kac voxel kenar atilsin (sinir artefakti: 1. kosuda %18)
TARGET_PER_T   = 200    # hedef T_pred/kare (T_true ~213)
MAX_PER_T      = 400    # target None ise tavan

VALIDATE = True         # train uzerinde recall olcumu
VAL_FRAMES = 12         # kac etiketli kare orneklensin (guvenilir recall icin)

## 1 — Veri erişimi (isim hardcode YOK)

In [ ]:
def open_image(zarr_path):
    """OME-Zarr goruntuyu ve olcegini ac. Olcek dosyadan okunur (hardcode etme)."""
    grp = zarr.open(str(zarr_path), mode='r')
    arr = grp['0']
    attrs = dict(grp.attrs)
    scale = None
    try:
        ms = attrs.get('multiscales') or attrs['ome']['multiscales']
        for tf in ms[0]['datasets'][0]['coordinateTransformations']:
            if tf.get('type') == 'scale':
                scale = tuple(float(v) for v in tf['scale'][-3:])   # (Z,Y,X)
    except Exception as e:
        print('  [uyari] olcek okunamadi:', e)
    if scale is None:
        scale = (1.625, 0.40625, 0.40625)   # yedek
    quant = (attrs.get('image_statistics') or {}).get('quantiles')
    return arr, np.array(scale), quant


# TEST ISIMLERI RUNTIME'DA BULUNUR — gizli set farkli isim/sayida gelir!
test_paths = sorted((ROOT / 'test').glob('*.zarr'))
print(f'{len(test_paths)} test ornegi bulundu:')
for p in test_paths:
    print('  ', p.stem)
assert test_paths, 'test/*.zarr bulunamadi!'

_arr, _sc, _q = open_image(test_paths[0])
print('\nshape', _arr.shape, '| dtype', _arr.dtype)
print('olcek (Z,Y,X) um/px:', _sc)
print('quantiles:', _q)

## 2 — Detection: anizotropik local-max

Z ekseni 4× kaba olduğu için bastırma penceresi **voxel'de anizotropik** olmalı:
5 µm → Z'de ~3 voxel, Y/X'te ~12 voxel. `maximum_filter` dikdörtgen pencerede ayrılabilir
olduğu için hızlı çalışır.

### 1. koşudan öğrenilen iki düzeltme

**(a) Eşik kare-başına olmalı.** Global `attrs['quantiles']['0.9']`=497 sabitini kullanınca
`T_pred` çöktü: `44b6_0b24845f`'te çeyrekler **2.8 → 17.4 → 58.4 → 67.7** node/kare (bazı
karelerde sıfır), `6bba_05db0fb1`'de tersine 190 → 117. Sebep: yoğunluk zamanla değişiyor
(photobleaching — EDA'da vardı). Artık eşik bir *kapı* değil, sadece gürültü tabanı;
asıl seçimi **kare başına en parlak N tepe** yapıyor → `T_pred` sabitleniyor.

**(b) Sınır artefaktı.** İlk koşuda node'ların **%18'i tam hacim sınırındaydı** (hacmin
%4.7'si sınır voxel'i → ~4× zenginleşme). `maximum_filter` kenarda pencereyi kırpıyor,
içeri yükselen gradyanın kenarı sahte local-max oluyor. `BORDER_MARGIN` ile eleniyor.

### Ceza hakkında — ölçüm sonrası düzeltme

Çarpan `max(0, 1 − 0.1·(T_pred−T_true)/T_true)` **zayıf**: `[0.9, 1.1]` aralığında.
2× fazla tahmin sadece %10 kaybettirir; ama tespit edilmeyen hücre jaccard'ı doğrudan
düşürür (FN). İlk koşuda çarpanımız 1.02–1.08'di (yani *bonus*) — skoru öldüren ceza değil,
**düşük recall**. Bu yüzden `T_true`'ya (~213/kare) kadar çıkmak doğru.

In [ ]:
def detect_centers(vol, scale, quant=None, target=TARGET_PER_T, max_n=MAX_PER_T):
    """(Z,Y,X) hacimde cekirdek merkezleri bul -> (N,3) voxel koordinat."""
    v = vol.astype(np.float32)

    # 1) yumusatma — sigma um cinsinden esit, voxel'de anizotropik
    sigma = SMOOTH_UM / scale
    sm = ndi.gaussian_filter(v, sigma=sigma)

    # 2) KARE-BASINA gurultu tabani.
    #    Global sabit esik (attrs quantile '0.9') KULLANMA: yogunluk zamanla degisiyor
    #    (photobleaching) -> karanlik karelerde T_pred sifira dusuyordu (2.8 node/kare!).
    #    Bu bir "kapi" degil, sadece taban; asil secimi (4) yapiyor.
    thr = float(np.percentile(sm, NOISE_PCT))

    # 3) anizotropik local-max: pencere yari-boyu ~SUPPRESS_UM
    half = np.maximum(1, np.round(SUPPRESS_UM / scale)).astype(int)   # (z,y,x) voxel
    size = tuple(2 * half + 1)
    mx = ndi.maximum_filter(sm, size=size, mode='nearest')
    peaks = (sm == mx) & (sm > thr)

    # 3b) SINIR ARTEFAKTI: maximum_filter kenarda pencereyi kirpiyor -> iceri yukselen
    #     gradyanin kenari sahte "local max" oluyor. Olculdu: node'larin %18'i tam
    #     sinirda (hacmin sadece %4.7'si sinir voxel'i). z=0'da sinyal zaten zayif.
    if BORDER_MARGIN > 0:
        m = BORDER_MARGIN
        keep_mask = np.zeros_like(peaks)
        keep_mask[m:-m, m:-m, m:-m] = True
        peaks &= keep_mask

    zz, yy, xx = np.nonzero(peaks)
    if len(zz) == 0:
        return np.empty((0, 3), dtype=int)
    vals = sm[zz, yy, xx]

    # 4) T_pred kontrolu: en parlak `target` tepe -> T_pred/kare sabitlenir.
    #    Ceza carpani [0.9, 1.1] araliginda: az tahmin recall'u oldurur (FN),
    #    biraz fazla tahmin ucuz. T_true'ya (~213/kare) kadar cikmakta fayda var.
    keep = min(len(vals), target if target is not None else max_n)
    if keep < len(vals):
        idx = np.argpartition(vals, -keep)[-keep:]     # argsort'tan hizli
    else:
        idx = np.arange(len(vals))
    return np.stack([zz[idx], yy[idx], xx[idx]], axis=1)


# --- Zaman boyunca tutarlilik kontrolu (eskiden burada 2.8 -> 68 rampa vardi) ---
arr, scale, quant = open_image(test_paths[0])
Zd, Yd, Xd = arr.shape[1:]
print('t   | T_pred | sinirda% | sn')
for t in [0, 30, 60, 99]:
    t0 = time.time()
    c = detect_centers(np.asarray(arr[t]), scale, quant)
    b = 0.0 if len(c) == 0 else float((
        (c[:, 0] == 0) | (c[:, 0] == Zd-1) |
        (c[:, 1] == 0) | (c[:, 1] == Yd-1) |
        (c[:, 2] == 0) | (c[:, 2] == Xd-1)).mean() * 100)
    print(f'{t:3d} | {len(c):6d} | {b:7.1f}% | {time.time()-t0:.2f}')
print('\nBeklenti: T_pred her karede ~TARGET_PER_T, sinirda% = 0.0')

## 3 — Detection kalitesi: train GT'ye karşı recall

GT **seyrek** (kare başına ~6 etiketli, gerçekte ~213 hücre). Bu yüzden **precision ölçülemez**
(etiketsiz hücreyi FP sayamayız) — ama **recall ölçülebilir**: her GT node'a 7 µm içinde
bir tahminimiz var mı? Detection'ın üst sınırını bu belirler.

In [ ]:
def load_geff(geff_path):
    """GEFF -> (nodes_df, edges (E,2)). Duz zarr ile; geff/tracksdata gerekmiyor."""
    g = zarr.open(str(geff_path), mode='r')
    nid = np.asarray(g['nodes/ids'])
    df = pd.DataFrame({
        'node_id': nid,
        't': np.asarray(g['nodes/props/t/values']),
        'z': np.asarray(g['nodes/props/z/values']),
        'y': np.asarray(g['nodes/props/y/values']),
        'x': np.asarray(g['nodes/props/x/values']),
    })
    return df, np.asarray(g['edges/ids'])


if VALIDATE:
    # her domainden 1 ornek (44b6 seyrek, 6bba yogun)
    val = []
    for pref in ('44b6', '6bba'):
        hits = sorted((ROOT / 'train').glob(f'{pref}_*.zarr'))
        if hits:
            val.append(hits[0])

    for zp in val:
        gp = zp.with_suffix('.geff')
        if not gp.exists():
            continue
        arr, scale, quant = open_image(zp)
        ndf, _ = load_geff(gp)
        # etiketli kareleri zamana YAY (ilk N degil) — yogunluk zamanla degisiyor
        ts_all = sorted(ndf.t.unique())
        ts = [ts_all[i] for i in np.linspace(0, len(ts_all)-1, min(VAL_FRAMES, len(ts_all))).astype(int)]

        tot = hit = 0
        npred, dists = [], []
        for t in ts:
            gt = ndf[ndf.t == t][['z', 'y', 'x']].to_numpy()
            pr = detect_centers(np.asarray(arr[int(t)]), scale, quant)
            npred.append(len(pr))
            tot += len(gt)
            if len(pr) == 0:
                continue
            d = np.linalg.norm((gt[:, None, :] - pr[None, :, :]) * scale, axis=2)  # um
            dmin = d.min(axis=1)
            hit += int((dmin <= 7.0).sum())
            dists.extend(dmin.tolist())

        med = float(np.median(dists)) if dists else float('nan')
        print(f'{zp.stem}: recall@7um = {hit}/{tot} = {hit/max(tot,1)*100:5.1f}%  '
              f'| T_pred/kare {int(np.min(npred))}-{int(np.max(npred))} (ort {int(np.mean(npred))})  '
              f'| GT->en yakin tahmin medyan {med:.2f} um')
    print('\nBeklenti: T_pred/kare araligi DAR olmali (eskiden 0-78 idi).')

## 4 — Linking: Hungarian, 8 µm gate

Ardışık kareler arasında µm cinsinden mesafe matrisi → `linear_sum_assignment`.
Gate'i aşan eşleşmeler atılır (hücre kayboldu/yeni girdi = appearance/disappearance).
**Gap-closing yok** — EDA soy içinde zamansal boşluk olmadığını gösterdi.

In [ ]:
def link_frames(prev_xyz, cur_xyz, scale, gate=LINK_GATE_UM):
    """prev/cur: (N,3) voxel. -> [(i_prev, j_cur), ...] gate icinde eslesmeler."""
    if len(prev_xyz) == 0 or len(cur_xyz) == 0:
        return []
    d = np.linalg.norm((prev_xyz[:, None, :] - cur_xyz[None, :, :]) * scale, axis=2)  # um
    # gate disini cok pahali yap ki Hungarian secmesin
    big = gate * 1000.0
    cost = np.where(d <= gate, d, big)
    ri, ci = linear_sum_assignment(cost)
    return [(int(i), int(j)) for i, j in zip(ri, ci) if d[i, j] <= gate]


def track_dataset(zarr_path, verbose=True):
    """Bir .zarr -> (nodes_df, edges_list). node_id dataset icinde 1'den ardisik."""
    arr, scale, quant = open_image(zarr_path)
    T = arr.shape[0]
    nodes, edges = [], []
    next_id = 1
    prev_xyz, prev_ids = np.empty((0, 3), int), []

    for t in range(T):
        cur = detect_centers(np.asarray(arr[t]), scale, quant)
        cur_ids = list(range(next_id, next_id + len(cur)))
        next_id += len(cur)
        for nid, (z, y, x) in zip(cur_ids, cur):
            nodes.append((nid, t, int(z), int(y), int(x)))
        for i, j in link_frames(prev_xyz, cur, scale):
            edges.append((prev_ids[i], cur_ids[j]))
        prev_xyz, prev_ids = cur, cur_ids
        if verbose and (t + 1) % 25 == 0:
            print(f'    t={t+1}/{T}  node={len(nodes)}  edge={len(edges)}')

    ndf = pd.DataFrame(nodes, columns=['node_id', 't', 'z', 'y', 'x'])
    return ndf, edges

## 5 — Tüm test setini işle → `submission.csv`

In [ ]:
rows = []
rid = 0
t_start = time.time()

for zp in test_paths:
    ds = zp.stem
    t0 = time.time()
    print(f'\n>>> {ds}')
    ndf, edges = track_dataset(zp)

    for r in ndf.itertuples(index=False):
        rows.append([rid, ds, 'node', r.node_id, r.t, r.z, r.y, r.x, -1, -1]); rid += 1
    for s, d in edges:
        rows.append([rid, ds, 'edge', -1, -1, -1, -1, -1, s, d]); rid += 1

    print(f'    BITTI: {len(ndf)} node, {len(edges)} edge, '
          f'{len(ndf)/max(ndf.t.nunique(),1):.0f} node/kare, {time.time()-t0:.0f} sn')

sub = pd.DataFrame(rows, columns=['id','dataset','row_type','node_id','t','z','y','x','source_id','target_id'])
sub.to_csv(OUT / 'submission.csv', index=False)
print(f'\n=== TOPLAM {len(sub)} satir, {time.time()-t_start:.0f} sn ===')
print(sub.head())

## 6 — Submit öncesi doğrulama

Submit kotası sınırlı — bozuk CSV yüzünden deneme yakmayalım.

In [ ]:
ref = pd.read_csv(ROOT / 'sample_submission.csv')
assert list(sub.columns) == list(ref.columns), 'kolonlar eslesmiyor!'

n, e = sub[sub.row_type == 'node'], sub[sub.row_type == 'edge']
ok = True
for ds, gn in n.groupby('dataset'):
    ge = e[e.dataset == ds]
    ids = set(gn.node_id)
    tmap = dict(zip(gn.node_id, gn.t))
    consec = sorted(ids) == list(range(1, len(ids) + 1))
    bad_ref = int((~ge.source_id.isin(ids)).sum() + (~ge.target_id.isin(ids)).sum())
    bad_dt = sum(1 for a, b in zip(ge.source_id, ge.target_id) if tmap.get(b, -99) - tmap.get(a, 0) != 1)
    print(f'{ds}: node 1..{max(ids)} ardisik={consec} | gecersiz-ref={bad_ref} | t+1-disi={bad_dt} '
          f'| T_pred/kare={len(gn)/gn.t.nunique():.0f}')
    ok &= consec and bad_ref == 0 and bad_dt == 0

print('\nHAZIR ✅  submission.csv gonderilebilir' if ok else '\nSORUN VAR ❌')
print('\nHatirlatma: Internet = OFF olmali, yoksa submit reddedilir.')

## 7 — Sonraki adımlar

Skor geldikten sonra, etkiye göre sıralı:

1. **`TARGET_PER_T` kalibrasyonu** — `T_true` bize verilmiyor; ~213/kare *tahmin*. Skoru
   150/200/250 ile karşılaştırıp fazla-tahmin cezasının nerede olduğunu ampirik bul.
2. **Detection kalitesi** — asıl darboğaz **instance ayrımı** (yerel SNR ~1.5×, çekirdekler
   temas halinde). Watershed veya çoklu-eşik hipotezleri dene; recall@7µm'yi ölç.
3. **Linking maliyeti** — sabit gate yerine mesafe + yoğunluk benzerliği; `pulp`/`cvxpy`
   ile küçük bir ILP (Ultrack mantığı) kurulabilir.
4. **Bölünme** (%10) — ebeveyn-kız ~6 µm; bir node'un iki çocuğa bağlanmasına izin ver.
5. **CV** — prefix-bazlı (`44b6`/`6bba`) fold; leaderboard'a değil kendi skoruna güven.

> **Süre riski:** gizli test 4'ten fazla film içerebilir. Yukarıdaki `sn/kare` ölçümünü
> kullanarak bütçe çıkar; gerekirse `SMOOTH_UM`/pencere boyutunu düşür.